# Notebook prerequisites

[Libpostal](https://github.com/openvenues/libpostal) is a C library for parsing addresses. The python bindings for it requires the local installation of the library and therefore so does this notebook.

To faciltate the matter and provide easy reproducibility, you will find a Dockerfile and docker compose file that sets up a jupyter server with libpostal installed. To run this notebook, you may simply run `docker-compose -f 'docker-compose.yml' up --build 'postal-jupyter` and connect to the URL in the logs (either in a browser to use jupyter lab GUI or by selecting it as the kernel in VS Code).

Alternatively the install instructions for libpostal are provided in [their github](https://github.com/openvenues/libpostal?tab=readme-ov-file#installation-maclinux) or the [python bindings' github](https://github.com/openvenues/pypostal#installation)

In [12]:
%pip install pandas postal rapidfuzz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 13.7 MB/s  0:00:00 eta 0:00:01
Note: you may need to restart the kernel to use updated packages.


# Testing libpostal

The following 3 code blocks test basic examples that libpostal can easily parse. Libposta parses addresses by segmenting them into the individual parts that represent different level's of the hierachical location. This is represented as a list of "address component", "component type" tuples.

In [1]:
from postal.parser import parse_address


parse_address('Berlin, Alexanderplatz 1, 10178')


[('berlin', 'house'),
 ('alexanderplatz', 'road'),
 ('1', 'house_number'),
 ('10178', 'postcode')]

In [2]:
parse_address('The Book Club 100-106 Leonard St, Shoreditch, London, Greater London, EC2A 4RH, United Kingdom')

[('the book club', 'house'),
 ('100-106', 'house_number'),
 ('leonard st', 'road'),
 ('shoreditch', 'suburb'),
 ('london', 'city'),
 ('greater london', 'state_district'),
 ('ec2a 4rh', 'postcode'),
 ('united kingdom', 'country')]

In [3]:
parse_address('The Book Club 100-106 Leonard St, London, EC2A 4RH, United Kingdom')

[('the book club', 'house'),
 ('100-106', 'house_number'),
 ('leonard st', 'road'),
 ('london', 'city'),
 ('ec2a 4rh', 'postcode'),
 ('united kingdom', 'country')]

As we can see the above addresses seem to be parsed correctly. Note that the component type "house" denotes the name of the building the address refers to, such as "The book club". This mostly applies to commercial or public service buildings, not residential addresses.

Now let us try parsing the addresses on a subset of the BZK dataset (dataset is not provided as it contains protected information).

In [7]:
import pandas as pd

df = pd.read_json("data/1870.json")

print(f"Number of BZK cards: {len(df)}")

sampled = df.sample(n=10)
sampled.sample(n=10)

Number of BZK cards: 1335


,CompensationOffice1,BZKNr,ApplicantFirstName,ApplicantLastName,ApplicantAltFirstName,ApplicantBirthName,ApplicantAltLastName,ApplicantBirthDate,ApplicantBirthPlace,ApplicantCurrentAddress,...,VictimBirthDate,VictimBirthPlace,VictimDeathDate,VictimDeathPlace,VictimCurrentAddress,Heirs,full_response,path,filename,model_name
59,Hess. Staatsministerium,24672,,,,,,,,,...,21.11.1870,"Lützelbach, Kreis Dieburg / Hessen",,,"Kelsterbach, Ringstr.17, Kreis Gr.-Gerau / Hessen",,"```python\n{\n ""CompensationOffice1"": ""Hess...",/home/bzk-data/1870/,1870_11_21_2_0.jpg,InternvlLmdeployModel_InternVL2_5-38B_p-9_fews...
1074,Bayerisches Landesentschädigungsamt,BEG 80 839,Maximo,Fechenbach,NaN,NaN,NaN,6.2.1870,Bad Mergentheim,"San Miguel (Prov. Buenos Aires), Caspar Campos...",...,"1.19.2.1924, 2.11.1.1897",NaN,31.12.1945,Konzentrationslager,NaN,None,"```python\n{\n ""CompensationOffice1"": ""Baye...",/home/bzk-data/1870/,1870_02_06_4_0.jpg,InternvlLmdeployModel_InternVL2_5-38B_p-9_fews...
88,Entschädigungsamt Berlin,162 104,Sophie,Stegemann,NaN,Buchthal,NaN,30.12.1870,NaN,"124 West, 90. Str., New York 24, N.Y./USA",...,27.11.1869,Berlin,NaN,NaN,NaN,5,"```python\n{\n ""CompensationOffice1"": ""Ents...",/home/bzk-data/1870/,1870_12_30_6_0.jpg,InternvlLmdeployModel_InternVL2_5-38B_p-9_fews...
324,Bayerisches Landesentschädigungsamt,BEG 41312,Franz,Laubmeier,NaN,NaN,NaN,29.8.1870,NaN,"München 25, Kistlerhofstr.95/0,r.",...,19.11.1911,NaN,11.9.1941,Russland,NaN,None,"```python\n{\n ""CompensationOffice1"": ""Baye...",/home/bzk-data/1870/,1870_08_29_2_0.jpg,InternvlLmdeployModel_InternVL2_5-38B_p-9_fews...
945,Regierungsbezirksamt für Wiedergutmachung und ...,39131,Lise,Gösing,NaN,NaN,NaN,4.7.1870,Tachau,"660 Stage ave., Apt. 2, Richmond, Calif.",...,24.1.1867,"Wien, Österreich",8.11.1943,Shanghai/China,NaN,None,"```python\n{\n ""CompensationOffice1"": ""Regi...",/home/bzk-data/1870/,1870_07_04_1_0.jpg,InternvlLmdeployModel_InternVL2_5-38B_p-9_fews...
448,Entschädigungsamt Berlin,50 392,Friedrich,Baumfeld,NaN,NaN,NaN,14.10.1870,NaN,"Epsom/Surrey, Woodcote House/England",...,NaN,NaN,NaN,NaN,NaN,None,"```python\n{\n ""CompensationOffice1"": ""Ents...",/home/bzk-data/1870/,1870_10_14_1_0.jpg,InternvlLmdeployModel_InternVL2_5-38B_p-9_fews...
361,Bezirksamt für Wiedergutmachung Köln,319333,Netti,Stern geb. Müller,NaN,NaN,NaN,1870,Tisovec,"Nath, München",...,NaN,NaN,NaN,NaN,NaN,None,"```python\n{\n ""CompensationOffice1"": ""Bezi...",/home/bzk-data/1870/,1870_00_00_42_0.jpg,InternvlLmdeployModel_InternVL2_5-38B_p-9_fews...
308,Landesamt für Wiedergutmachung Bremen,A.Z. 3957,,,,,,,,,...,28.12.1870,Trebnitz/Schlesien,14.8.1942,Theresienstadt,,,"```python\n{\n ""CompensationOffice1"": ""Land...",/home/bzk-data/1870/,1870_12_28_6_0.jpg,InternvlLmdeployModel_InternVL2_5-38B_p-9_fews...
1319,Entschädigungsbehörde Köln,Art. V-56-II-Nr. 92 769,Hana,Gottlieb,NaN,NaN,NaN,24.2.1970,Tokai/Ungarn,"Ramat Gan, Amudar Nr. 45/Isreal",...,NaN,NaN,NaN,NaN,NaN,None,"```python\n{\n ""CompensationOffice1"": ""Ents...",/home/bzk-data/1870/,1870_02_24_2_0.jpg,InternvlLmdeployModel_InternVL2_5-38B_p-9_fews...
1009,"Hansestadt Hamburg, Sozialbehörde, Amt für Wie...",5767/BC4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,21.9.70,Klosterallee 78,NaN,NaN,NaN,None,"```python\n{\n ""CompensationOffice1"": ""Hans...",/home/bzk-data/1870/,1870_09_21_1_0.jpg,InternvlLmdeployModel_InternVL2_5-38B_p-9_fews...


In [15]:
PLACE_COLS = [
    "ApplicantBirthPlace", 
    "ApplicantCurrentAddress", 
    "VictimBirthPlace", 
    "VictimDeathPlace",
    "VictimCurrentAddress"
]

# List of labels from libpostal - labels and descriptions obtained from https://github.com/openvenues/libpostal/blob/master/README.md#parser-labels
ADDRESS_COMPONENT_TYPES = [
    "house", # venue name e.g. "Brooklyn Academy of Music", and building names e.g. "Empire State Building"
    "category", # for category queries like "restaurants", etc.
    "near", # phrases like "in", "near", etc. used after a category phrase to help with parsing queries like "restaurants in Brooklyn"
    "house_number", # usually refers to the external (street-facing) building number. In some countries this may be a compount, hyphenated number which also includes an apartment number, or a block number (a la Japan), but libpostal will just call it the house_number for simplicity.
    "road", # street name(s)
    "unit", # an apartment, unit, office, lot, or other secondary unit designator
    "level", # expressions indicating a floor number e.g. "3rd Floor", "Ground Floor", etc.
    "staircase", # numbered/lettered staircase
    "entrance", # numbered/lettered entrance
    "po_box", # post office box: typically found in non-physical (mail-only) addresses
    "postcode", # postal codes used for mail sorting
    "suburb", # usually an unofficial neighborhood name like "Harlem", "South Bronx", or "Crown Heights"
    "city_district", # these are usually boroughs or districts within a city that serve some official purpose e.g. "Brooklyn" or "Hackney" or "Bratislava IV"
    "city", # any human settlement including cities, towns, villages, hamlets, localities, etc.
    "island", # named islands e.g. "Maui"
    "state_district", # usually a second-level administrative division or county.
    "state", # a first-level administrative division. Scotland, Northern Ireland, Wales, and England in the UK are mapped to "state" as well (convention used in OSM, GeoPlanet, etc.)
    "country_region", # informal subdivision of a country without any political status
    "country", # sovereign nations and their dependent territories, anything with an ISO-3166 code.
    "world_region", # currently only used for appending “West Indies” after the country name, a pattern frequently used in the English-speaking Caribbean e.g. “Jamaica, West Indies”
] 


In [21]:
def parse_address_to_series(address):
    parsed = pd.Series(pd.NA, index=ADDRESS_COMPONENT_TYPES, dtype="string")
    if pd.isna(address) or address.strip() == "":
        return parsed
    for component, label in parse_address(address, language="de"):
        if label not in ADDRESS_COMPONENT_TYPES:
            raise ValueError(f"Unknown address component label: {label}")
        parsed[label] = component
    return parsed

parsed_df = pd.DataFrame(columns=pd.MultiIndex.from_product([PLACE_COLS, ADDRESS_COMPONENT_TYPES]))

for col in PLACE_COLS:
    parsed_df[col] = df[col].apply(parse_address_to_series)

parsed_df = parsed_df.stack(level=0).dropna(how='all', axis=1)

for col in parsed_df.columns:
    print(f"Example value for col {col}: {parsed_df[col].dropna().sample(n=1, random_state=42).values[0]}")


original_addresses = df[PLACE_COLS].stack(level=0).dropna()
original_addresses.name = "originalAddress"
sampled = original_addresses.sample(n=20, random_state=42).to_frame().merge(parsed_df, left_index=True, right_index=True).dropna(how="all", axis=1).fillna("")
sampled

Example value for col house: gleichenoie thüringen
Example value for col category: libur
Example value for col near: bei
Example value for col house_number: 5109
Example value for col road: 9939-65th road
Example value for col unit: apt. 16e
Example value for col level: rindelbach krs
Example value for col po_box: apartado 52
Example value for col postcode: 6639
Example value for col suburb: jassy
Example value for col city_district: bronx
Example value for col city: piotrikow/polen
Example value for col state_district: côte d'or
Example value for col state: n.y.
Example value for col country: us


,,originalAddress,house,house_number,road,unit,city,country
495,ApplicantBirthPlace,Putnok Ungarn,,,,,putnok,ungarn
470,ApplicantCurrentAddress,"Buenos Aires, Argentina, Las Heras 2928",argentina,2928,las heras,,buenos aires,
914,ApplicantBirthPlace,,,,,,,
616,ApplicantCurrentAddress,"25, 8, 67 Kli",,67,kli,,,
440,VictimDeathPlace,Löger Hamburg,löger,,,,hamburg,
717,VictimCurrentAddress,"München-Harlaching, Petristrasse 7",münchen-harlaching,7,petristrasse,,,
120,ApplicantBirthPlace,Kieserzschbracht,kieserzschbracht,,,,,
364,VictimDeathPlace,Auschwitz,,,,,auschwitz,
1048,ApplicantCurrentAddress,"Haifa, Israel, Mapu Str. Beth Hrim",,,haifa israel mapu str. beth hrim,,,
679,VictimBirthPlace,Warschau,,,,,warschau,


Here we note many addresses that are parsed incorrectly. In particular, libpostal incorrectly identifies:

- "Argentina" as a house instead of a country
- "Frankfurt/Main" as a house instead of a city
- "München-Harlaching" as a house instead of a city
- "polen" as part of the city name instead of the country

Additionally, the address "Santiago de Chile, San Francisco `XXX`, Casa `X` USA" (numbers redacted for data protection) seems to be parsed incorrectly, identifying "santiago de chile san franscisco" as the street name, not separating "San Franscisco" as city name, likely due to the unusual address format with what seems to be a house number after San Francisco.

But upon further analysis, the address itself may be incorrect. Searching on Google Maps for "Santiago de Chile" in the American city "San Franscisco" yields no results. However searching for "San Francisco `XXX`" in the city of "Santiago" in "Chile" points to a house on google maps. It may be possible that the address refers to this place rather than any location on the USA and USA as the country was incorrectly written down posteriorly or exctracted by the MLLM. The usage of "Casa" in spanish instead of "House" in english to indicate the apartment number corrobates this possibility. (**TODO** look at the card to see if anything else suggests this possibility, row 1001)

Aside this problem, the other issues still provide an easier format to work with. It seems that the type "house" acts as a fallback for components that could not be identified as anything else. If we compare the term in house to a lookup table of location names we might be able to solve these misclassifications. The only other misclassification in the sample that could not be solved this way would be "/ polen" as a part of the city name.

In [10]:
df.iloc[1001]

CompensationOffice1                                 Entschädigungsamt Berlin
BZKNr                                                                160 648
ApplicantFirstName                                                     Minna
ApplicantLastName                                                     Hirsch
ApplicantAltFirstName                                                    NaN
ApplicantBirthName                                                      Aron
ApplicantAltLastName                                                     NaN
ApplicantBirthDate                                                14.11.1870
ApplicantBirthPlace                                               Stenschewo
ApplicantCurrentAddress     Santiago de Chile, San Francisco 366, Casa 5 USA
ApplicantMaritalStatus                                                Mutter
VictimFirstName                                                          Leo
VictimLastName                                                        Hirsch

In [23]:
from rapidfuzz import process
from rapidfuzz.distance import DamerauLevenshtein
MIN_SIMILARITY = 0.85

place_name_lookup = pd.read_csv("place-name-lookup.csv")

def try_solve_house(row):
    if pd.isna(row['house']) or row['house'].strip() == "":
        return row
    matches = process.extract(row["house"], place_name_lookup["searchKey"], scorer=DamerauLevenshtein.normalized_similarity, score_cutoff=MIN_SIMILARITY)
    if len(matches) > 0:
        row = row.copy()
        best_match = matches[0]
        matched_place = place_name_lookup.iloc[best_match[2]]
        row[matched_place['placeType']] = row['house']
        row['house'] = pd.NA
    return row

parsed_df2 = parsed_df.apply(try_solve_house, axis=1)

sampled = original_addresses.sample(n=20, random_state=42).to_frame().merge(parsed_df2, left_index=True, right_index=True).dropna(how="all", axis=1).fillna("")
sampled

,,originalAddress,house,house_number,road,unit,city,country
495,ApplicantBirthPlace,Putnok Ungarn,,,,,putnok,ungarn
470,ApplicantCurrentAddress,"Buenos Aires, Argentina, Las Heras 2928",,2928,las heras,,buenos aires,argentina
914,ApplicantBirthPlace,,,,,,,
616,ApplicantCurrentAddress,"25, 8, 67 Kli",,67,kli,,,
440,VictimDeathPlace,Löger Hamburg,löger,,,,hamburg,
717,VictimCurrentAddress,"München-Harlaching, Petristrasse 7",münchen-harlaching,7,petristrasse,,,
120,ApplicantBirthPlace,Kieserzschbracht,kieserzschbracht,,,,,
364,VictimDeathPlace,Auschwitz,,,,,auschwitz,
1048,ApplicantCurrentAddress,"Haifa, Israel, Mapu Str. Beth Hrim",,,haifa israel mapu str. beth hrim,,,
679,VictimBirthPlace,Warschau,,,,,warschau,
